In [ ]:
# Install required packages
!pip install tensorflow-hub

In [ ]:
# Imports and setup
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers, Model, Input
import numpy as np

# Set random seed
tf.random.set_seed(42)
np.random.seed(42)

# Load pretrained MobileViT-Small
MOBILEVIT_URL = "https://tfhub.dev/google/imagenet/mobilevit_small/classification/2"
mobilevit_base = hub.KerasLayer(MOBILEVIT_URL, trainable=False)  # Freeze pretrained weights

In [ ]:
# Helper function to extract intermediate features from MobileViT
def build_mobilevit_encoder(input_shape=(256, 256, 3)):
    """Build MobileViT encoder and return intermediate features for skip connections."""
    # Create base model with dummy input to extract features
    inputs = Input(input_shape)
    
    # Create intermediate model to extract features
    x = inputs
    features = []
    
    # MobileViT stages (simplified for demonstration)
    # Stage 1
    x = layers.Conv2D(32, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    features.append(x)  # 128x128
    
    # Stage 2
    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    features.append(x)  # 64x64
    
    # Load pretrained MobileViT
    mobilevit = tf.keras.Sequential([
        hub.KerasLayer(MOBILEVIT_URL, trainable=False),
    ])
    
    # Get remaining features from MobileViT
    x = mobilevit(x)
    features.append(x)  # 32x32
    
    # Create model
    model = Model(inputs, features)
    return model

In [ ]:
# Adaptive attention decoder block
def adaptive_attention_decoder(x, skip_connection, out_channels, num_heads=4):
    """Decoder block with adaptive attention mechanism."""
    # Upsample
    x = layers.Conv2DTranspose(out_channels, 2, strides=2, padding='same')(x)
    
    # Concatenate skip connection
    if skip_connection is not None:
        x = layers.Concatenate()([x, skip_connection])
    
    # Local features
    local_feat = layers.Conv2D(out_channels, 3, padding='same')(x)
    local_feat = layers.BatchNormalization()(local_feat)
    local_feat = layers.Activation('relu')(local_feat)
    
    # Multi-head self attention
    # Reshape to sequence
    h, w = tf.shape(x)[1], tf.shape(x)[2]
    seq = layers.Reshape((h * w, out_channels))(local_feat)
    
    # Apply attention
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=out_channels // num_heads
    )(seq, seq)
    
    # Reshape back
    attn_out = layers.Reshape((h, w, out_channels))(attn_out)
    
    # Adaptive gating
    gate = layers.Conv2D(out_channels, 1, activation='sigmoid')(local_feat)
    attended = layers.Multiply()([attn_out, gate])
    
    # Combine with input
    x = layers.Add()([local_feat, attended])
    x = layers.LayerNormalization()(x)
    
    # Final conv block
    x = layers.Conv2D(out_channels, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    return x

In [ ]:
# Build complete model with pretrained MobileViT encoder
def build_mobilevit_unet(input_shape=(256, 256, 3), num_classes=1):
    """Build U-Net with pretrained MobileViT encoder and adaptive attention decoder."""
    inputs = Input(input_shape)
    
    # Encoder (pretrained MobileViT)
    encoder = build_mobilevit_encoder(input_shape)
    skip_features = encoder(inputs)
    
    # Bridge
    x = skip_features[-1]
    
    # Decoder with adaptive attention
    # Assuming skip_features contains features at different scales
    decoder_filters = [256, 128, 64, 32]
    
    for i, filters in enumerate(decoder_filters):
        if i < len(skip_features) - 1:
            skip = skip_features[-(i+2)]
        else:
            skip = None
        x = adaptive_attention_decoder(x, skip, filters)
    
    # Output
    if num_classes == 1:
        outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    else:
        outputs = layers.Conv2D(num_classes, 1, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='MobileViT_UNet')
    return model

# Metrics and loss functions
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

# Combined loss
def combined_loss(y_true, y_pred):
    return 0.5 * tf.keras.losses.binary_crossentropy(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

In [ ]:
# Build and compile model
model = build_mobilevit_unet(input_shape=(256, 256, 3), num_classes=1)

# Compile with metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[dice_coef, 'accuracy']
)

# Print model summary
model.summary()

# Test with dummy data
x = np.random.rand(1, 256, 256, 3).astype('float32')
y = model.predict(x)
print('\nOutput shape:', y.shape)

# Model Comparison: MobileViT-UNet vs Standard U-Net

We'll compare:
1. Parameter count
2. Inference speed (FPS)
3. Dice coefficient
4. Memory usage

This gives us a clear view of the trade-offs between accuracy and computational efficiency.

In [ ]:
# Standard U-Net implementation for comparison
def build_standard_unet(input_shape=(256, 256, 3), num_classes=1):
    inputs = Input(input_shape)
    
    # Encoder
    conv1 = layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)
    
    conv2 = layers.Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)
    
    conv3 = layers.Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)
    
    conv4 = layers.Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = layers.Conv2D(512, 3, activation='relu', padding='same')(conv4)
    pool4 = layers.MaxPooling2D(pool_size=(2, 2))(conv4)
    
    # Bridge
    conv5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(pool4)
    conv5 = layers.Conv2D(1024, 3, activation='relu', padding='same')(conv5)
    
    # Decoder
    up6 = layers.Conv2DTranspose(512, 2, strides=(2, 2), padding='same')(conv5)
    concat6 = layers.Concatenate()([up6, conv4])
    conv6 = layers.Conv2D(512, 3, activation='relu', padding='same')(concat6)
    conv6 = layers.Conv2D(512, 3, activation='relu', padding='same')(conv6)
    
    up7 = layers.Conv2DTranspose(256, 2, strides=(2, 2), padding='same')(conv6)
    concat7 = layers.Concatenate()([up7, conv3])
    conv7 = layers.Conv2D(256, 3, activation='relu', padding='same')(concat7)
    conv7 = layers.Conv2D(256, 3, activation='relu', padding='same')(conv7)
    
    up8 = layers.Conv2DTranspose(128, 2, strides=(2, 2), padding='same')(conv7)
    concat8 = layers.Concatenate()([up8, conv2])
    conv8 = layers.Conv2D(128, 3, activation='relu', padding='same')(concat8)
    conv8 = layers.Conv2D(128, 3, activation='relu', padding='same')(conv8)
    
    up9 = layers.Conv2DTranspose(64, 2, strides=(2, 2), padding='same')(conv8)
    concat9 = layers.Concatenate()([up9, conv1])
    conv9 = layers.Conv2D(64, 3, activation='relu', padding='same')(concat9)
    conv9 = layers.Conv2D(64, 3, activation='relu', padding='same')(conv9)
    
    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid')(conv9)
    
    model = Model(inputs=inputs, outputs=outputs, name='Standard_UNet')
    return model

In [ ]:
# Benchmark utilities
import time
import psutil
import numpy as np
from tensorflow.keras.utils import Sequence

# Simple data generator for benchmarking
class BenchmarkDataGenerator(Sequence):
    def __init__(self, batch_size=8, img_size=(256, 256), num_batches=10):
        self.batch_size = batch_size
        self.img_size = img_size
        self.num_batches = num_batches
    
    def __len__(self):
        return self.num_batches
    
    def __getitem__(self, idx):
        # Generate random images and masks
        x = np.random.rand(self.batch_size, *self.img_size, 3).astype('float32')
        y = (np.random.rand(self.batch_size, *self.img_size, 1) > 0.5).astype('float32')
        return x, y

def measure_inference_time(model, input_shape=(256, 256, 3), num_runs=50):
    # Warm up
    x = np.random.rand(1, *input_shape).astype('float32')
    for _ in range(5):
        model.predict(x)
    
    # Measure
    times = []
    for _ in range(num_runs):
        start = time.time()
        model.predict(x)
        times.append(time.time() - start)
    
    avg_time = np.mean(times)
    fps = 1.0 / avg_time
    return avg_time, fps

def get_model_size(model):
    return np.sum([np.prod(v.get_shape().as_list()) for v in model.trainable_variables])

In [ ]:
# Run benchmarks
input_shape = (256, 256, 3)
batch_size = 8

# Initialize models
mobilevit_unet = build_mobilevit_unet(input_shape=input_shape)
standard_unet = build_standard_unet(input_shape=input_shape)

# Compile both models
for model in [mobilevit_unet, standard_unet]:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=combined_loss,
        metrics=[dice_coef]
    )

# Create benchmark dataset
bench_generator = BenchmarkDataGenerator(batch_size=batch_size, img_size=input_shape[:2])

# Compare parameter counts
print("Parameter Count Comparison:")
print(f"MobileViT-UNet: {get_model_size(mobilevit_unet):,} parameters")
print(f"Standard U-Net: {get_model_size(standard_unet):,} parameters")

# Compare inference speed
print("\nInference Speed Comparison:")
for model, name in [(mobilevit_unet, "MobileViT-UNet"), (standard_unet, "Standard U-Net")]:
    avg_time, fps = measure_inference_time(model, input_shape)
    print(f"{name}:")
    print(f"  Average inference time: {avg_time*1000:.2f} ms")
    print(f"  Frames per second: {fps:.2f}")

# Compare Dice coefficient on test batch
print("\nDice Coefficient Comparison:")
test_batch = next(iter(bench_generator))
for model, name in [(mobilevit_unet, "MobileViT-UNet"), (standard_unet, "Standard U-Net")]:
    metrics = model.evaluate(*test_batch, verbose=0)
    print(f"{name} Dice score: {metrics[1]:.4f}")

# Adaptive MobileViT U‑Net (Hybrid CNN–Transformer)

This notebook implements an adaptive hybrid U‑Net that injects a small MobileViT‑style
transformer block into decoder stages. The block uses patch embedding + a lightweight
transformer encoder (Keras MultiHeadAttention) and an adaptive gating mechanism to
control how much global attention contributes vs local convolutional features.

Replace the placeholder data loading with ISIC 2020 or Cityscapes loaders to train on
real data. The notebook includes model build / demo cells for a quick sanity check.

In [ ]:
# Imports and setup
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import numpy as np
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Basic conv / encoder / decoder helpers
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def down_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D()(c)
    return c, p

def up_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

In [ ]:
# MobileViT-style patch embedding + lightweight transformer encoder (implemented simply)
def patch_embedding(x, patch_size=2, embed_dim=64):
    # x: (B,H,W,C) -> unfold into non-overlapping patches of size patch_size
    H = tf.shape(x)[1]
    W = tf.shape(x)[2]
    # Use a conv with stride=patch_size to create patch vectors
    x = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='valid')(x)
    # now (B, H/ps, W/ps, embed_dim) -> flatten to sequence later in transformer block
    return x

def lightweight_transformer_on_patches(x, num_heads=4, mlp_dim=128, dropout=0.0):
    # x: (B, H_p, W_p, embed_dim) -> treat as sequence of length H_p*W_p
    shape = tf.shape(x)
    B, Hp, Wp, C = shape[0], shape[1], shape[2], shape[3]
    seq = layers.Reshape((Hp * Wp, C))(x)
    # MHSA (queries/keys/values all from seq)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=max(1, C // num_heads), dropout=dropout)(seq, seq)
    seq = layers.Add()([seq, attn])
    seq = layers.LayerNormalization(epsilon=1e-6)(seq)
    # MLP
    ff = layers.Dense(mlp_dim, activation='gelu')(seq)
    ff = layers.Dense(C)(ff)
    seq = layers.Add()([seq, ff])
    seq = layers.LayerNormalization(epsilon=1e-6)(seq)
    # reshape back to (B, Hp, Wp, C)
    out = layers.Reshape((Hp, Wp, C))(seq)
    return out

def mobilevit_block(x, patch_size=2, embed_dim=64, num_heads=4, mlp_dim=128):
    # Local representation (conv)
    local_feat = layers.Conv2D(embed_dim, 3, padding='same', activation='relu')(x)
    # Patch embedding over local features
    patches = patch_embedding(local_feat, patch_size=patch_size, embed_dim=embed_dim)
    # Transformer on patches
    transformed = lightweight_transformer_on_patches(patches, num_heads=num_heads, mlp_dim=mlp_dim)
    # Upsample transformed patches to original resolution by pixel shuffle-like expansion
    # naive resize using nearest/neighbour for simplicity
    Hp = tf.shape(transformed)[1]
    Wp = tf.shape(transformed)[2]
    # resize to match local_feat spatial dims
    transformed_up = tf.image.resize(transformed, (tf.shape(local_feat)[1], tf.shape(local_feat)[2]), method='bilinear')
    # fuse local and global features, then return
    fused = layers.Concatenate()([local_feat, transformed_up])
    fused = layers.Conv2D(tf.shape(x)[-1], 1, padding='same', activation='relu')(fused)
    # adaptive gate (learn how much to use fused/global info)
    gate = layers.Conv2D(1, 1, activation='sigmoid')(fused)
    out = layers.Multiply()([fused, gate])
    out = layers.Add()([x, out])
    return out

In [ ]:
# Build a U-Net with MobileViT blocks in the decoder
def build_mobilevit_unet(input_shape=(128,128,1), base_filters=32, patch_size=2, embed_dim=64):
    inputs = Input(input_shape)
    c1, p1 = down_block(inputs, base_filters)
    c2, p2 = down_block(p1, base_filters*2)
    c3, p3 = down_block(p2, base_filters*4)
    c4, p4 = down_block(p3, base_filters*8)
    b = conv_block(p4, base_filters*16)
    # decoder with MobileViT blocks injected
    u6 = layers.Conv2DTranspose(base_filters*8, 2, strides=2, padding='same')(b)
    u6 = layers.Concatenate()([u6, c4])
    u6 = mobilevit_block(u6, patch_size=patch_size, embed_dim=embed_dim)
    u6 = conv_block(u6, base_filters*8)

    u7 = layers.Conv2DTranspose(base_filters*4, 2, strides=2, padding='same')(u6)
    u7 = layers.Concatenate()([u7, c3])
    u7 = mobilevit_block(u7, patch_size=patch_size, embed_dim=embed_dim)
    u7 = conv_block(u7, base_filters*4)

    u8 = layers.Conv2DTranspose(base_filters*2, 2, strides=2, padding='same')(u7)
    u8 = layers.Concatenate()([u8, c2])
    u8 = mobilevit_block(u8, patch_size=patch_size, embed_dim=embed_dim)
    u8 = conv_block(u8, base_filters*2)

    u9 = layers.Conv2DTranspose(base_filters, 2, strides=2, padding='same')(u8)
    u9 = layers.Concatenate()([u9, c1])
    u9 = mobilevit_block(u9, patch_size=patch_size, embed_dim=embed_dim)
    u9 = conv_block(u9, base_filters)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(u9)
    model = Model(inputs, outputs, name='MobileViT_UNet')
    return model

In [ ]:
# Losses and metrics
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.*intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [ ]:
# Quick sanity build + forward pass on dummy data
model = build_mobilevit_unet(input_shape=(128,128,1), base_filters=16, patch_size=2, embed_dim=64)
model.summary()
x = np.random.rand(1,128,128,1).astype('float32')
y = model.predict(x)
print('Output shape:', y.shape)